In [1]:
from pathlib import Path
import subprocess

repo = Path("/content/SimpleMem")

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/aiming-lab/SimpleMem.git", str(repo)],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Commit:", commit)

for name in ["requirements.txt", "config.py.example", "test_locomo10.py", "locomo10.json"]:
    matches = sorted(path.relative_to(repo) for path in repo.rglob(name))
    locations = ", ".join(str(path) for path in matches)
    print(f"{name}: {locations or 'Not found'}")

Commit: db80b6a7c591e0ea730a058e9f5fc4eb06572299
requirements.txt: EvolveMem/requirements.txt, MCP/reference/requirements.txt, MCP/requirements.txt, OmniSimpleMem/requirements.txt, SKILL/simplemem-skill/requirements.txt, requirements.txt, simplemem/integrations/reference/requirements.txt, simplemem/integrations/requirements.txt, simplemem/integrations/simplemem-skill/requirements.txt
config.py.example: MCP/reference/config.py.example, SKILL/simplemem-skill/src/config.py.example, config.py.example, simplemem/integrations/reference/config.py.example, simplemem/integrations/simplemem-skill/src/config.py.example
test_locomo10.py: MCP/reference/test_locomo10.py, simplemem/integrations/reference/test_locomo10.py, test_locomo10.py
locomo10.json: data/locomo10.json


In [2]:
import json
from pathlib import Path
from urllib.request import urlopen

repo = Path("/content/SimpleMem")
data_dir = repo / "data"
data_dir.mkdir(exist_ok=True)

data_path = data_dir / "locomo10.json"
data_url = "https://raw.githubusercontent.com/snap-research/locomo/main/data/locomo10.json"

if data_path.exists():
    samples = json.loads(data_path.read_bytes())
else:
    with urlopen(data_url, timeout=60) as response:
        raw_data = response.read()
    samples = json.loads(raw_data)
    data_path.write_bytes(raw_data)

first_sample = samples[0]
first_qa = first_sample["qa"][0]

print("Dataset path:", data_path)
print("Conversations:", len(samples))
print("First sample ID:", first_sample["sample_id"])
print("Questions in first sample:", len(first_sample["qa"]))
print("First question:", first_qa["question"])
print("Reference answer:", first_qa.get("answer", "No reference answer"))

Dataset path: /content/SimpleMem/data/locomo10.json
Conversations: 10
First sample ID: conv-26
Questions in first sample: 199
First question: When did Caroline go to the LGBTQ support group?
Reference answer: 7 May 2023


In [3]:
import json
from pathlib import Path

data_dir = Path("/content/SimpleMem/data")
source_path = data_dir / "locomo10.json"
test_path = data_dir / "locomo_smoke_1q.json"

samples = json.loads(source_path.read_text(encoding="utf-8"))
sample = samples[0].copy()
sample["qa"] = sample["qa"][:1]

test_path.write_text(
    json.dumps([sample], ensure_ascii=False, indent=2),
    encoding="utf-8",
)

saved_samples = json.loads(test_path.read_text(encoding="utf-8"))
assert saved_samples[0]["conversation"] == samples[0]["conversation"]
assert len(saved_samples[0]["qa"]) == 1

print("Test file:", test_path)
print("Sample ID:", saved_samples[0]["sample_id"])
print("Questions:", len(saved_samples[0]["qa"]))

Test file: /content/SimpleMem/data/locomo_smoke_1q.json
Sample ID: conv-26
Questions: 1


In [4]:
%cd /content/SimpleMem
%pip install -r requirements.txt

/content/SimpleMem


In [5]:
from pathlib import Path
from importlib.metadata import version

repo = Path("/content/SimpleMem")
print("Project ready:", (repo / "test_locomo10.py").is_file())
print("Test data ready:", (repo / "data" / "locomo_smoke_1q.json").is_file())

from openai import OpenAI

print("OpenAI client ready:", version("openai"))

Project ready: True
Test data ready: True
OpenAI client ready: 2.3.0


In [9]:
import os
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("DEEPSEEK_API_KEY").strip()
base_url = input("API base URL: ").strip().rstrip("/")
model = input("Model name: ").strip()

client = OpenAI(
    api_key=api_key,
    base_url=base_url,
    timeout=60.0,
    max_retries=0,
)

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Reply with the single word OK."}],
)

os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_BASE_URL"] = base_url

print("Model:", response.model)
print("Reply:", response.choices[0].message.content)

API base URL: https://api.deepseek.com
Model name: deepseek-v4-flash
Model: deepseek-flash
Reply: OK


In [10]:
import ast
from pathlib import Path

repo = Path("/content/SimpleMem")
config_source = (repo / "config.py.example").read_text(encoding="utf-8")
test_source = (repo / "test_locomo10.py").read_text(encoding="utf-8")

print("Config fields:")
for node in ast.walk(ast.parse(config_source)):
    if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
        if any(word in node.id for word in ("API", "URL", "MODEL", "EMBEDDING")):
            print(node.id)

print("\nEvaluation arguments:")
for node in ast.walk(ast.parse(test_source)):
    if isinstance(node, ast.Call) and getattr(node.func, "attr", "") == "add_argument":
        print(ast.unparse(node))

Config fields:
OPENAI_API_KEY
OPENAI_BASE_URL
LLM_MODEL
EMBEDDING_MODEL
EMBEDDING_DIMENSION
EMBEDDING_CONTEXT_LENGTH
JUDGE_API_KEY
JUDGE_BASE_URL
JUDGE_MODEL

Evaluation arguments:
parser.add_argument('--dataset', type=str, default='test_ref/locomo10.json', help='Path to LoComo10 dataset')
parser.add_argument('--num-samples', type=int, default=None, help='Number of samples to test (default: all)')
parser.add_argument('--no-save', action='store_true', help='Do not save results to file')
parser.add_argument('--result-file', type=str, default='locomo10_test_results.json', help='Path to the result file')
parser.add_argument('--parallel-questions', action='store_true', help='Enable parallel processing of questions within each sample')
parser.add_argument('--llm-judge', action='store_true', help='Enable LLM-as-judge evaluation for semantic answer comparison')
parser.add_argument('--test-workers', type=int, default=None, help='Number of parallel workers for question testing (default: use conf

In [11]:
import os
from pathlib import Path

repo = Path("/content/SimpleMem")

if not os.environ.get("OPENAI_API_KEY") or not os.environ.get("OPENAI_BASE_URL"):
    raise RuntimeError("Run the successful DeepSeek API test cell again.")

template = (repo / "config.py.example").read_text(encoding="utf-8")

overrides = """
import os

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = os.environ["OPENAI_BASE_URL"]
LLM_MODEL = "deepseek-v4-flash"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
EMBEDDING_DIMENSION = 1024
"""

(repo / "config.py").write_text(template + "\n" + overrides, encoding="utf-8")
print("Config ready.")

Config ready.


In [12]:
import json
import subprocess
import sys
from datetime import datetime, timezone

dataset = repo / "data" / "locomo_smoke_1q.json"
output_dir = repo / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir.mkdir(parents=True)

result_path = output_dir / "result.json"
log_path = output_dir / "run.log"

command = [
    sys.executable, "-u", "test_locomo10.py",
    "--dataset", str(dataset),
    "--num-samples", "1",
    "--result-file", str(result_path),
]

print("Running one question. Log:", log_path, flush=True)

with log_path.open("w", encoding="utf-8") as log:
    run = subprocess.run(
        command,
        cwd=repo,
        stdout=log,
        stderr=subprocess.STDOUT,
    )

print("Exit code:", run.returncode)
print("Result exists:", result_path.is_file())

if run.returncode == 0 and result_path.is_file():
    result = json.loads(result_path.read_text(encoding="utf-8"))
    print("Result file:", result_path)
    print("Result preview:")
    print(json.dumps(result, indent=2, ensure_ascii=False)[:3000])
else:
    print("Last 40 log lines:")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-40:]))

Running one question. Log: /content/SimpleMem/runs/20261112T065117797633Z/run.log
Exit code: 1
Result exists: False
Last 40 log lines:
    import tensorflow as tf
  File "/usr/local/lib/python3.13/dist-packages/tensorflow/__init__.py", line 49, in <module>
    from tensorflow._api.v2 import __internal__
  File "/usr/local/lib/python3.13/dist-packages/tensorflow/_api/v2/__internal__/__init__.py", line 8, in <module>
    from tensorflow._api.v2.__internal__ import autograph
  File "/usr/local/lib/python3.13/dist-packages/tensorflow/_api/v2/__internal__/autograph/__init__.py", line 8, in <module>
    from tensorflow.python.autograph.core.ag_ctx import control_status_ctx # line: 34
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/tensorflow/python/autograph/core/ag_ctx.py", line 21, in <module>
    from tensorflow.python.autograph.utils import ag_logging
  File "/usr/local/lib/python3.13/dist-packages/tensorflow/pyth

In [13]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path("/content/SimpleMem")
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

check = subprocess.run(
    [sys.executable, "test_locomo10.py", "--help"],
    cwd=repo,
    capture_output=True,
    text=True,
)

print("Startup exit code:", check.returncode)

if check.returncode == 0:
    print("Evaluation script loaded successfully.")
else:
    print(check.stderr)

Startup exit code: 0
Evaluation script loaded successfully.


In [14]:
import nltk

nltk.download("punkt_tab", download_dir="/root/nltk_data", raise_on_error=True)
print("Tokenizer check:", nltk.word_tokenize("7 May 2023"))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Tokenizer check: ['7', 'May', '2023']


In [18]:
import json
from pathlib import Path

runs_dir = Path("/content/SimpleMem/runs")
result_files = sorted(runs_dir.glob("*/result.json"), reverse=True)

if not result_files:
    print("No result files found in the current runtime.")

for path in result_files:
    result = json.loads(path.read_text(encoding="utf-8"))
    count = result["summary"]["num_questions"]
    print(f"{count} question(s) | {path}")

No result files found in the current runtime.


In [19]:
from pathlib import Path

repo = Path("/content/SimpleMem")
print("Evaluation script:", (repo / "test_locomo10.py").is_file())
print("Config:", (repo / "config.py").is_file())
print("Full dataset:", (repo / "data" / "locomo10.json").is_file())
print("20-question dataset:", (repo / "data" / "locomo_conv26_seed42_20q.json").is_file())

Evaluation script: True
Config: True
Full dataset: True
20-question dataset: False


In [20]:
import json
import random
from pathlib import Path

repo = Path("/content/SimpleMem")
data_dir = repo / "data"
samples = json.loads((data_dir / "locomo10.json").read_text(encoding="utf-8"))
sample = next(item for item in samples if item["sample_id"] == "conv-26")

eligible_indices = [
    index
    for index, qa in enumerate(sample["qa"])
    if index > 0 and qa.get("answer") is not None and str(qa["answer"]).strip()
]
selected_indices = sorted(random.Random(42).sample(eligible_indices, 20))
selected_qa = [sample["qa"][index] for index in selected_indices]

dataset = data_dir / "locomo_conv26_seed42_20q.json"
subset = {**sample, "qa": selected_qa}
dataset.write_text(json.dumps([subset], indent=2, ensure_ascii=False), encoding="utf-8")

print("Dataset:", dataset)
print("Sample ID:", sample["sample_id"])
print("Questions:", len(selected_qa))
print("Original question indices:", selected_indices)

Dataset: /content/SimpleMem/data/locomo_conv26_seed42_20q.json
Sample ID: conv-26
Questions: 20
Original question indices: [7, 8, 9, 23, 24, 27, 29, 36, 51, 56, 58, 60, 63, 71, 108, 109, 130, 140, 144, 167]


In [21]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import nltk

repo = Path("/content/SimpleMem")
dataset = repo / "data" / "locomo_conv26_seed42_20q.json"

samples = json.loads(dataset.read_text(encoding="utf-8"))
assert len(samples) == 1 and samples[0]["sample_id"] == "conv-26"
assert len(samples[0]["qa"]) == 20

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

if not os.environ.get("OPENAI_API_KEY") or not os.environ.get("OPENAI_BASE_URL"):
    raise RuntimeError("Run the DeepSeek API test cell first.")

nltk.download("punkt_tab", quiet=True, raise_on_error=True)
nltk.word_tokenize("Tokenizer check.")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = repo / "runs" / f"baseline_20q_{stamp}"
output_dir.mkdir(parents=True)

result_path = output_dir / "result.json"
log_path = output_dir / "run.log"

command = [
    sys.executable, "-u", "test_locomo10.py",
    "--dataset", str(dataset),
    "--num-samples", "1",
    "--result-file", str(result_path),
]

print("Running the 20-question baseline.")
print("Log:", log_path, flush=True)

with log_path.open("w", encoding="utf-8") as log:
    run = subprocess.run(command, cwd=repo, stdout=log, stderr=subprocess.STDOUT)

print("Exit code:", run.returncode)

if run.returncode == 0 and result_path.is_file():
    result = json.loads(result_path.read_text(encoding="utf-8"))
    summary = result["summary"]
    overall = result["aggregated_metrics"]["overall"]

    print("Questions evaluated:", summary["num_questions"])
    print("Mean F1:", round(overall["f1"]["mean"], 4))
    print("Exact match:", round(overall["exact_match"]["mean"], 4))
    print("Mean retrieval time (s):", round(summary["avg_retrieval_time"], 2))
    print("Result file:", result_path)
else:
    print("Last 40 log lines:")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-40:]))

Running the 20-question baseline.
Log: /content/SimpleMem/runs/baseline_20q_20261112T070651465368Z/run.log
Exit code: 0
Questions evaluated: 20
Mean F1: 0.4175
Exact match: 0.2
Mean retrieval time (s): 24.63
Result file: /content/SimpleMem/runs/baseline_20q_20261112T070651465368Z/result.json


In [22]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

repo = Path("/content/SimpleMem")
baseline_path = repo / "runs" / "baseline_20q_20261112T070651465368Z" / "result.json"
dataset_path = repo / "data" / "locomo_conv26_seed42_20q.json"
archive_path = Path("/content/simplemem_baseline_20q.zip")

with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(baseline_path, "baseline_result.json")
    archive.write(dataset_path, "locomo_conv26_seed42_20q.json")

print("Retrieval settings:")
for line in (repo / "config.py").read_text(encoding="utf-8").splitlines():
    setting = line.split("#", 1)[0].strip()
    if "TOP_K" in setting and "=" in setting:
        print(setting)

files.download(str(archive_path))

Retrieval settings:
SEMANTIC_TOP_K = 25
KEYWORD_TOP_K = 5
STRUCTURED_TOP_K = 5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Trial 1

In [23]:
import re
from pathlib import Path

repo = Path("/content/SimpleMem")
config_path = repo / "config.py"
source = config_path.read_text(encoding="utf-8")

pattern = r"(?m)^(SEMANTIC_TOP_K\s*=\s*)(\d+)([ \t]*(?:#[^\n]*)?)$"
matches = list(re.finditer(pattern, source))

if len(matches) != 1:
    raise ValueError("Expected exactly one SEMANTIC_TOP_K assignment.")

match = matches[0]
old_value = int(match.group(2))

if old_value not in (25, 10):
    raise ValueError(f"Unexpected SEMANTIC_TOP_K value: {old_value}")

updated = source[:match.start(2)] + "10" + source[match.end(2):]
config_path.write_text(updated, encoding="utf-8")

print(f"SEMANTIC_TOP_K: {old_value} -> 10")
for line in updated.splitlines():
    if line.startswith(("KEYWORD_TOP_K =", "STRUCTURED_TOP_K =")):
        print(line)

SEMANTIC_TOP_K: 25 -> 10
KEYWORD_TOP_K = 5
STRUCTURED_TOP_K = 5


In [24]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import nltk

repo = Path("/content/SimpleMem")
dataset = repo / "data" / "locomo_conv26_seed42_20q.json"

samples = json.loads(dataset.read_text(encoding="utf-8"))
assert len(samples) == 1 and samples[0]["sample_id"] == "conv-26"
assert len(samples[0]["qa"]) == 20

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

if not os.environ.get("OPENAI_API_KEY") or not os.environ.get("OPENAI_BASE_URL"):
    raise RuntimeError("Run the DeepSeek API test cell first.")

nltk.download("punkt_tab", quiet=True, raise_on_error=True)
nltk.word_tokenize("Tokenizer check.")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = repo / "runs" / f"semantic_top10_20q_{stamp}"
output_dir.mkdir(parents=True)

result_path = output_dir / "result.json"
log_path = output_dir / "run.log"

command = [
    sys.executable, "-u", "test_locomo10.py",
    "--dataset", str(dataset),
    "--num-samples", "1",
    "--result-file", str(result_path),
]

print("Running the 20-question experiment with SEMANTIC_TOP_K=10.")
print("Log:", log_path, flush=True)

with log_path.open("w", encoding="utf-8") as log:
    run = subprocess.run(command, cwd=repo, stdout=log, stderr=subprocess.STDOUT)

print("Exit code:", run.returncode)

if run.returncode == 0 and result_path.is_file():
    result = json.loads(result_path.read_text(encoding="utf-8"))
    summary = result["summary"]
    overall = result["aggregated_metrics"]["overall"]

    print("Questions evaluated:", summary["num_questions"])
    print("Mean F1:", round(overall["f1"]["mean"], 4))
    print("Exact match:", round(overall["exact_match"]["mean"], 4))
    print("Mean retrieval time (s):", round(summary["avg_retrieval_time"], 2))
    print("Result file:", result_path)
else:
    print("Last 40 log lines:")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-40:]))

Running the 20-question experiment with SEMANTIC_TOP_K=10.
Log: /content/SimpleMem/runs/semantic_top10_20q_20261112T075056399383Z/run.log
Exit code: 0
Questions evaluated: 20
Mean F1: 0.4806
Exact match: 0.25
Mean retrieval time (s): 24.12
Result file: /content/SimpleMem/runs/semantic_top10_20q_20261112T075056399383Z/result.json


In [25]:
import json
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
from google.colab import files

repo = Path("/content/SimpleMem")
runs = repo / "runs"
baseline_path = runs / "baseline_20q_20261112T070651465368Z" / "result.json"
modified_path = runs / "semantic_top10_20q_20261112T075056399383Z" / "result.json"
dataset_path = repo / "data" / "locomo_conv26_seed42_20q.json"

def load_answers(path, label):
    result = json.loads(path.read_text(encoding="utf-8"))
    rows = [
        {
            "question": item["question"],
            "reference": item["reference"],
            f"answer_{label}": item["answer"],
            f"f1_{label}": item["metrics"]["f1"],
        }
        for item in result["detailed_results"]
    ]
    return pd.DataFrame(rows)

comparison = load_answers(baseline_path, "top25").merge(
    load_answers(modified_path, "top10"),
    on=["question", "reference"],
    validate="one_to_one",
)
assert len(comparison) == 20, "Expected 20 matched questions."

comparison["f1_change"] = comparison["f1_top10"] - comparison["f1_top25"]
comparison = comparison.sort_values("f1_change", ascending=False)

csv_path = Path("/content/simplemem_question_comparison.csv")
comparison.to_csv(csv_path, index=False)

archive_path = Path("/content/simplemem_20q_comparison.zip")
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(baseline_path, "baseline_top25.json")
    archive.write(modified_path, "modified_top10.json")
    archive.write(dataset_path, dataset_path.name)
    archive.write(csv_path, csv_path.name)

for label, row in [
    ("Largest F1 gain", comparison.iloc[0]),
    ("Lowest F1 change", comparison.iloc[-1]),
]:
    print(f"\n{label}: {row['f1_change']:+.4f}")
    print("Question:", row["question"])
    print("Reference:", row["reference"])
    print("Top-25 answer:", row["answer_top25"])
    print("Top-10 answer:", row["answer_top10"])

files.download(str(archive_path))


Largest F1 gain: +0.8462
Question: What did Mel and her kids make during the pottery workshop?
Reference: pots
Top-25 answer: They all made pots, and her kids made a cup with a dog face.
Top-10 answer: pots

Lowest F1 change: -0.4114
Question: When did Caroline meet up with her friends, family, and mentors?
Reference: The week before 9 June 2023
Top-25 answer: In the week prior to 09 June 2023
Top-10 answer: Between 26 June 2023 and 2 July 2023 for a picnic with friends and family; no combined mentors meetup is specified.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Trial 2

In [27]:
import json
import os
import runpy
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import nltk

# Set the project and dataset paths.
repo = Path("/content/SimpleMem")
dataset = repo / "data" / "locomo_conv26_seed42_20q.json"
config_path = repo / "config.py"

# Use the same runtime settings as Trial 1.
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

if not os.environ.get("OPENAI_API_KEY") or not os.environ.get("OPENAI_BASE_URL"):
    raise RuntimeError("Run the DeepSeek API test cell first.")

# Verify the actual retrieval settings before starting the experiment.
settings = runpy.run_path(str(config_path))
expected_settings = {
    "SEMANTIC_TOP_K": 10,
    "KEYWORD_TOP_K": 5,
    "STRUCTURED_TOP_K": 5,
}

for name, expected in expected_settings.items():
    actual = settings.get(name)
    if actual != expected:
        raise ValueError(f"{name}: expected {expected}, found {actual}.")

# Verify the fixed 20-question dataset.
samples = json.loads(dataset.read_text(encoding="utf-8"))
assert len(samples) == 1, "Expected one conversation."
assert samples[0]["sample_id"] == "conv-26", "Expected conv-26."
assert len(samples[0]["qa"]) == 20, "Expected 20 questions."

try:
    nltk.data.find("tokenizers/punkt_tab/english/")
except LookupError:
    nltk.download("punkt_tab", quiet=True, raise_on_error=True)

nltk.word_tokenize("Tokenizer check.")

# Create a separate output directory for this run.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = repo / "runs" / f"semantic_top10_trial2_{stamp}"
output_dir.mkdir(parents=True)

result_path = output_dir / "top10_trial2.json"
log_path = output_dir / "run.log"

command = [
    sys.executable, "-u", "test_locomo10.py",
    "--dataset", str(dataset),
    "--num-samples", "1",
    "--result-file", str(result_path),
]

print("Starting Top-10 Trial 2 | conv-26 | 20 questions", flush=True)
print("Settings verified: semantic=10, keyword=5, structured=5", flush=True)
print("Log:", log_path, flush=True)

with log_path.open("w", encoding="utf-8") as log:
    run = subprocess.run(command, cwd=repo, stdout=log, stderr=subprocess.STDOUT)

print("Exit code:", run.returncode)

if run.returncode != 0 or not result_path.is_file():
    print("Last 40 log lines:")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-40:]))
    raise RuntimeError("Evaluation did not finish successfully. See the log above.")

# Check and summarize the completed run.
result = json.loads(result_path.read_text(encoding="utf-8"))
summary = result["summary"]
overall = result["aggregated_metrics"]["overall"]
assert summary["num_questions"] == 20, "Expected 20 evaluated questions."

print("Questions evaluated:", summary["num_questions"])
print("Mean F1:", round(overall["f1"]["mean"], 4))
print("Exact match:", round(overall["exact_match"]["mean"], 4))
print("Mean retrieval time (s):", round(summary["avg_retrieval_time"], 2))
print("Result file:", result_path)

# Download a backup before the Colab session ends.
from google.colab import files

files.download(str(result_path))

Starting Top-10 Trial 2 | conv-26 | 20 questions
Settings verified: semantic=10, keyword=5, structured=5
Log: /content/SimpleMem/runs/semantic_top10_trial2_20261112T090824726899Z/run.log
Exit code: 0
Questions evaluated: 20
Mean F1: 0.4829
Exact match: 0.25
Mean retrieval time (s): 20.14
Result file: /content/SimpleMem/runs/semantic_top10_trial2_20261112T090824726899Z/top10_trial2.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Top‑25 Trial 2

In [28]:
import json
import os
import re
import runpy
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from google.colab import files

repo = Path("/content/SimpleMem")
config_path = repo / "config.py"
dataset = repo / "data" / "locomo_conv26_seed42_20q.json"

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

if not os.environ.get("OPENAI_API_KEY") or not os.environ.get("OPENAI_BASE_URL"):
    raise RuntimeError("Run the DeepSeek API test cell first.")

samples = json.loads(dataset.read_text(encoding="utf-8"))
assert len(samples) == 1 and samples[0]["sample_id"] == "conv-26"
assert len(samples[0]["qa"]) == 20

# Change only the semantic retrieval setting.
source = config_path.read_text(encoding="utf-8")
pattern = r"(?m)^(SEMANTIC_TOP_K[ \t]*=[ \t]*)(?:10|25)\b"
updated, count = re.subn(pattern, r"\g<1>25", source)
if count != 1:
    raise ValueError("Expected one SEMANTIC_TOP_K assignment set to 10 or 25.")
config_path.write_text(updated, encoding="utf-8")

settings = runpy.run_path(str(config_path))
assert settings["SEMANTIC_TOP_K"] == 25
assert settings["KEYWORD_TOP_K"] == 5
assert settings["STRUCTURED_TOP_K"] == 5

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = repo / "runs" / f"semantic_top25_trial2_{stamp}"
output_dir.mkdir(parents=True)

result_path = output_dir / "top25_trial2.json"
log_path = output_dir / "run.log"

command = [
    sys.executable, "-u", "test_locomo10.py",
    "--dataset", str(dataset),
    "--num-samples", "1",
    "--result-file", str(result_path),
]

print("Starting Top-25 Trial 2 | conv-26 | 20 questions", flush=True)
print("Settings verified: semantic=25, keyword=5, structured=5", flush=True)
print("Log:", log_path, flush=True)

with log_path.open("w", encoding="utf-8") as log:
    run = subprocess.run(command, cwd=repo, stdout=log, stderr=subprocess.STDOUT)

print("Exit code:", run.returncode)

if run.returncode != 0 or not result_path.is_file():
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-40:]))
    raise RuntimeError("Evaluation did not finish successfully.")

result = json.loads(result_path.read_text(encoding="utf-8"))
summary = result["summary"]
overall = result["aggregated_metrics"]["overall"]
assert summary["num_questions"] == 20, "Expected 20 evaluated questions."

print("Questions evaluated:", summary["num_questions"])
print("Mean F1:", round(overall["f1"]["mean"], 4))
print("Exact match:", round(overall["exact_match"]["mean"], 4))
print("Mean retrieval time (s):", round(summary["avg_retrieval_time"], 2))
print("Result file:", result_path)

files.download(str(result_path))

Starting Top-25 Trial 2 | conv-26 | 20 questions
Settings verified: semantic=25, keyword=5, structured=5
Log: /content/SimpleMem/runs/semantic_top25_trial2_20261112T093411864111Z/run.log
Exit code: 0
Questions evaluated: 20
Mean F1: 0.4335
Exact match: 0.2
Mean retrieval time (s): 22.17
Result file: /content/SimpleMem/runs/semantic_top25_trial2_20261112T093411864111Z/top25_trial2.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Trial 3

In [29]:
import json
import os
import re
import runpy
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

from google.colab import files

repo = Path("/content/SimpleMem")
config_path = repo / "config.py"
dataset = repo / "data" / "locomo_conv26_seed42_20q.json"
output_dir = repo / "runs" / "trial3_conv26_seed42"
output_dir.mkdir(parents=True, exist_ok=True)

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
if not os.environ.get("OPENAI_API_KEY") or not os.environ.get("OPENAI_BASE_URL"):
    raise RuntimeError("Run the DeepSeek API test cell first.")

samples = json.loads(dataset.read_text(encoding="utf-8"))
assert len(samples) == 1 and samples[0]["sample_id"] == "conv-26"
expected_questions = [qa["question"] for qa in samples[0]["qa"]]
assert len(expected_questions) == 20

for top_k in (10, 25):
    name = f"top{top_k}_trial3"
    result_path = output_dir / f"{name}.json"
    log_path = output_dir / f"{name}.log"

    if not result_path.exists():
        source = config_path.read_text(encoding="utf-8")
        updated, count = re.subn(
            r"(?m)^(SEMANTIC_TOP_K[ \t]*=[ \t]*)(?:10|25)\b",
            rf"\g<1>{top_k}",
            source,
        )
        if count != 1:
            raise ValueError("Expected one SEMANTIC_TOP_K assignment.")
        config_path.write_text(updated, encoding="utf-8")

        settings = runpy.run_path(str(config_path))
        assert settings["SEMANTIC_TOP_K"] == top_k
        assert settings["KEYWORD_TOP_K"] == settings["STRUCTURED_TOP_K"] == 5

        pending_path = output_dir / f"{name}.pending.json"
        pending_path.unlink(missing_ok=True)
        print(f"Starting {name}. Log: {log_path}", flush=True)

        with log_path.open("w", encoding="utf-8") as log:
            run = subprocess.run(
                [
                    sys.executable, "-u", "test_locomo10.py",
                    "--dataset", str(dataset),
                    "--num-samples", "1",
                    "--result-file", str(pending_path),
                ],
                cwd=repo,
                stdout=log,
                stderr=subprocess.STDOUT,
            )

        if run.returncode != 0 or not pending_path.is_file():
            print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-40:]))
            raise RuntimeError(f"{name} did not finish. Check the log.")

        read_path = pending_path
    else:
        print(f"Checking saved result: {name}", flush=True)
        read_path = result_path

    result = json.loads(read_path.read_text(encoding="utf-8"))
    rows = result["detailed_results"]
    assert result["summary"]["num_questions"] == 20
    assert [row["question"] for row in rows] == expected_questions

    if read_path != result_path:
        read_path.replace(result_path)

    overall = result["aggregated_metrics"]["overall"]
    print(f"{name} complete | Questions: {len(rows)}")
    print(f"Mean F1: {overall['f1']['mean']:.4f}")
    print(f"Exact match: {overall['exact_match']['mean']:.2f}")
    print(f"Mean retrieval time: {result['summary']['avg_retrieval_time']:.2f} s")
    print(flush=True)

archive_path = Path("/content/simplemem_trial3.zip")
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for top_k in (10, 25):
        path = output_dir / f"top{top_k}_trial3.json"
        archive.write(path, path.name)

files.download(str(archive_path))

Starting top10_trial3. Log: /content/SimpleMem/runs/trial3_conv26_seed42/top10_trial3.log
top10_trial3 complete | Questions: 20
Mean F1: 0.4425
Exact match: 0.20
Mean retrieval time: 22.59 s

Starting top25_trial3. Log: /content/SimpleMem/runs/trial3_conv26_seed42/top25_trial3.log
top25_trial3 complete | Questions: 20
Mean F1: 0.5072
Exact match: 0.25
Mean retrieval time: 20.11 s



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>